# Día 9 · Unidad 9 — Fuentes de datos (I): bases de datos relacionales y SQLModel

**Objetivos de aprendizaje**
- Explicar qué es una base de datos y por qué una hoja de cálculo o un CSV no siempre basta.
- Distinguir entre base de datos en archivo único (SQLite) y base de datos en servidor (PostgreSQL, MySQL).
- Entender el modelo relacional: tablas, `primary key`, `foreign key`.
- Reconocer el riesgo de SQL injection y por qué un ORM lo evita por defecto.
- Definir tu primer modelo de datos con `SQLModel` (construido sobre `SQLAlchemy` y `Pydantic`).

**Duración estimada:** 60-75 minutos.

**Requisitos previos:** Día 5 (clases, atributos, `dataclasses`) — un modelo `SQLModel` es, por dentro, una clase de Python con anotaciones de tipo.


## 1. ¿Por qué una base de datos y no un CSV?

Hasta ahora habéis trabajado con datos en archivos planos (`data/raw/cartera_polizas.csv`) cargados enteros en memoria con pandas. Funciona bien para análisis puntual, pero como sistema de gestión de datos tiene límites claros:

- **Concurrencia**: si dos procesos escriben en el mismo CSV a la vez, uno puede sobrescribir al otro.
- **Consistencia**: si un mismo dato (p. ej. la provincia de un asegurado) aparece repetido en varias filas, una actualización a medias puede dejar el archivo en un estado contradictorio.
- **Tamaño**: cargar un CSV entero en memoria para leer una sola fila es un desperdicio cuando la tabla tiene millones de registros.

Una **base de datos** es un sistema diseñado específicamente para almacenar, consultar y modificar datos estructurados de forma eficiente y segura frente a estos problemas.

### BD en archivo único vs. BD en servidor

- **Archivo único** (p. ej. **SQLite**): toda la base de datos vive en un solo archivo (`asegurados.db`), gestionado directamente por una librería que se ejecuta dentro de tu propio proceso Python. No hay ningún servidor aparte que arrancar. Es el motor que usaremos en este notebook — viene preinstalado con Python, así que no requiere nada adicional.
- **Servidor** (p. ej. **PostgreSQL**, **MySQL**): la base de datos es una aplicación aparte, normalmente en otra máquina, con la que tu código habla por red. Tu programa nunca lee ni escribe los archivos de datos directamente — se los pide a la aplicación de base de datos, que es quien de verdad gestiona el almacenamiento. Es el modelo habitual en producción, porque soporta muchos más usuarios concurrentes.

`SQLAlchemy` (y `SQLModel`, que se construye encima) hablan el mismo lenguaje Python para ambos casos — cambiar de SQLite a PostgreSQL en un proyecto real es, en gran medida, cambiar una URL de conexión, no reescribir el código.

> 📊 En Excel esto sería la diferencia entre un único archivo `.xlsx` que abres tú solo en tu portátil, y una base de datos compartida en la que varias personas trabajan a la vez sin pisarse: Excel no está pensado para lo segundo (de ahí los líos de "el archivo está bloqueado por otro usuario"), una base de datos de servidor sí.


## 2. El modelo relacional

Imaginad que queremos guardar los asegurados de una cartera de pólizas. La opción más simple sería una única tabla:

| id | nombre | producto | provincia | comunidad |
|---|---|---|---|---|
| 1 | Anna Puig | AUTO | barcelona | cataluña |
| 2 | Marc Soler | HOGAR | tarragona | cataluña |
| 3 | Laura Ruiz | VIDA | madrid | madrid |

Funciona, pero tiene un problema de fondo: la provincia y la comunidad autónoma están **repetidas** en cada fila. Si mañana "cataluña" pasa a escribirse "catalunya" en el sistema, hay que actualizar todas las filas que la mencionan — y si el proceso falla a mitad, la tabla queda en un estado inconsistente (algunas filas actualizadas, otras no).

El **modelo relacional** resuelve esto separando los datos en varias tablas, cada una con una responsabilidad clara, y conectándolas mediante identificadores:


In [ ]:
import pandas as pd

# Tabla de provincias: cada provincia aparece UNA sola vez
provincias = pd.DataFrame({
    "id": [1, 2, 3],
    "provincia": ["barcelona", "tarragona", "madrid"],
    "comunidad": ["cataluña", "cataluña", "madrid"],
})

# Tabla de asegurados: en vez de repetir "cataluña", guarda solo el id de la provincia
asegurados = pd.DataFrame({
    "id": [1, 2, 3],
    "nombre": ["Anna Puig", "Marc Soler", "Laura Ruiz"],
    "producto": ["AUTO", "HOGAR", "VIDA"],
    "provincia_id": [1, 1, 3],
})

provincias


In [ ]:
asegurados


Ahora "cataluña" existe en un único sitio (`provincias`). Cambiarlo es una sola actualización, y es imposible que dos filas de asegurados queden con nombres distintos para la misma provincia.

### Primary key y foreign key

- **Primary key (PK)**: la columna que identifica de forma única cada fila de una tabla. En `provincias`, es `id`. Ninguna base de datos SQL te deja tener dos filas con la misma primary key.
- **Foreign key (FK)**: una columna que hace referencia a la primary key de **otra** tabla. En `asegurados`, `provincia_id` es una foreign key que apunta a `provincias.id`. Es el mecanismo que conecta las tablas entre sí sin duplicar datos.

Cada tabla, en la jerga técnica de bases de datos SQL, se llama también **relación** — de ahí el nombre "base de datos relacional". (La palabra no se refiere a la relación *entre* tablas, sino a la tabla en sí misma: es un término histórico que confunde a mucha gente la primera vez.)

> ⚠️ **Error típico**: pensar que "relacional" significa "con relaciones entre tablas" (foreign keys). En realidad el término viene de las matemáticas (teoría de conjuntos) y designa simplemente a la tabla. Que las tablas se conecten mediante FKs es una consecuencia práctica del diseño, no lo que da nombre al modelo.

### SQL

El lenguaje que se diseñó para consultar y modificar este modelo se llama **SQL** (*Structured Query Language*). Cada motor de base de datos (SQLite, PostgreSQL, MySQL...) soporta una variante del mismo lenguaje base. Por ejemplo, para leer todos los asegurados:

```sql
SELECT * FROM asegurados;
```

o solo los de la provincia con `id = 1`:

```sql
SELECT * FROM asegurados WHERE provincia_id = 1;
```

No vais a escribir SQL a mano en este curso — para eso está `SQLModel` — pero conviene reconocerlo, porque es el lenguaje común detrás de cualquier ORM y el que veréis en herramientas de BI, en logs de bases de datos, o en la documentación de casi cualquier sistema con el que trabajéis.


## 3. El problema de mezclar SQL con Python a mano: SQL injection

La forma más directa de ejecutar SQL desde Python sería construir la instrucción como un string y enviarla tal cual:

```python
asegurado_id = input("Escribe el id del asegurado: ")
statement = f"SELECT * FROM asegurados WHERE id = {asegurado_id};"
resultado = conexion.execute(statement)
```

El problema: si `asegurado_id` viene de un usuario externo (un formulario web, un parámetro de API...), nada impide que alguien escriba algo como:

```
1; DROP TABLE asegurados;
```

y el string final que se envía a la base de datos sea:

```sql
SELECT * FROM asegurados WHERE id = 1; DROP TABLE asegurados;
```

Esto se llama **SQL injection**, y es una de las vulnerabilidades más comunes (y más destructivas) en aplicaciones que tocan bases de datos. La defensa se llama **SQL sanitization**: asegurarse de que cualquier valor que venga de fuera se trate siempre como un **dato**, nunca como parte del código SQL.

> ⚠️ **Error típico**: construir SQL con f-strings o `%` a partir de datos de usuario. Es exactamente el patrón que abre la puerta a SQL injection. La regla general: si estás construyendo SQL con concatenación de strings a partir de un valor que no controlas tú, hazlo con un ORM o con parámetros preparados (`?`/`:nombre`), nunca a mano.

Un **ORM** (*Object-Relational Mapper*) evita este problema por diseño: en vez de escribir SQL como texto, escribes código Python (clases, objetos, comparaciones), y es el ORM quien traduce eso a SQL de forma segura por debajo, sanitizando automáticamente cualquier valor que le pases.


## 4. SQLModel: tu primer modelo de datos

`SQLModel` es la librería que usaremos en este curso para hablar con bases de datos SQL desde Python. Está construida sobre dos librerías que quizás ya conocéis de nombre:

- **SQLAlchemy**: el ORM más usado del ecosistema Python — motor de conexión, traducción a SQL, gestión de sesiones.
- **Pydantic**: validación de datos con anotaciones de tipo — la misma idea que ya usasteis en el Día 5 con `dataclasses`, pero con validación en tiempo de ejecución.

Un modelo `SQLModel` es, por dentro, una clase normal de Python — con la diferencia de que además sabe representar una tabla:


In [ ]:
from typing import Optional
from sqlmodel import SQLModel, Field


class Asegurado(SQLModel):
    '''Representa un asegurado de la cartera (sin tabla asociada todavía).'''

    id: Optional[int] = Field(default=None, primary_key=True)
    asegurado_id: str
    nombre: str
    edad: int
    provincia: str


# Como cualquier clase de Python: se instancia igual, con validación de tipos incluida
anna = Asegurado(asegurado_id="AS20001", nombre="Anna Puig", edad=42, provincia="barcelona")
anna


Fíjate en el paralelismo con `dataclasses` (Día 5): en vez de escribir `__init__` a mano, se definen los campos con anotaciones de tipo y `SQLModel` genera el resto. La diferencia clave es que estas clases, cuando se marcan con `table=True` (lo veremos en el siguiente notebook), representan directamente una tabla de la base de datos — cada instancia es una fila.

`SQLModel` sanitiza automáticamente cualquier valor que le paséis al construir una consulta, así que el riesgo de SQL injection del apartado anterior desaparece por diseño mientras uséis su API (y no construyáis SQL en crudo con f-strings, claro).

**Para ti:** define una clase `Poliza` (sin tabla todavía, igual que `Asegurado` arriba) con los campos `id` (opcional, futura primary key), `poliza_id: str`, `producto: str` y `prima_anual: float`. No hace falta que la conectes a nada todavía — solo practica la sintaxis.


In [ ]:
# Tu código aquí: clase Poliza


## Resumen

- Una base de datos gestiona datos estructurados de forma más robusta que un CSV suelto: concurrencia, consistencia y eficiencia con grandes volúmenes.
- SQLite es una base de datos en archivo único que corre dentro de tu propio proceso Python (sin servidor); PostgreSQL/MySQL son bases de datos de servidor, el modelo habitual en producción.
- El **modelo relacional** organiza los datos en varias tablas conectadas por `primary key` (identifica de forma única cada fila) y `foreign key` (referencia a la PK de otra tabla), evitando duplicar información.
- SQL es el lenguaje estándar (con variantes por motor) para consultar y modificar bases de datos relacionales.
- Construir SQL concatenando strings con datos externos abre la puerta a **SQL injection**; un **ORM** como SQLModel sanitiza los valores automáticamente porque nunca construyes SQL como texto a mano.
- `SQLModel` = `SQLAlchemy` (motor ORM) + `Pydantic` (validación con tipos): una clase de Python con anotaciones de tipo que representa una tabla.

**Siguiente:** `02_teoria_bbdd_crud.ipynb`, donde creamos la base de datos de verdad, con datos de `cartera_polizas.csv`, y completamos el CRUD (Create, Read, Update, Delete).
